# Lab 02 – Data Preprocessing, Feature Engineering and Feature Selection
## In-Class Guided Notebook: Titanic Passenger Survival

This notebook is designed for a **live step-by-step demonstration**.

### How to use it during class
- Basic setup, variable names and code structure are already active.
- Only the important lines that introduce a new preprocessing or modelling decision are commented.
- Teaching steps are marked using a double-commented `## TODO:` line.
- Select the commented code lines below the relevant TODO and press **Ctrl + /** on Windows/Linux or **Cmd + /** on macOS.
- The `## TODO:` line may also be selected; it becomes `# TODO:` and remains a valid comment.
- Run the cell after uncommenting the required code.

This allows the instructor to explain each decision while gradually completing the code.  
Do not learn imputation, scaling, encoding or feature-selection parameters from the full dataset before the train–test split.


**Guided version:** Every major output is followed by an interpretation, expected values, and the main concept to understand.


## 0. Setup

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

In [ ]:
DATA_PATH = Path("titanic.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Place titanic.csv in the same folder as this notebook.")

df_raw = pd.read_csv(DATA_PATH)
display(df_raw.head())
print("Shape:", df_raw.shape)

## Warm-up – Diagnose and repair a tiny dirty dataset

This five-row example isolates problems that are easier to explain than in a large file:

- inconsistent categories (`Male`, `male`, `M`);
- an impossible negative age;
- mixed height units (`cm` and `m`);
- a duplicate record.

Reveal one cleaning decision at a time and inspect the result after each step.


In [ ]:
dirty_demo = pd.DataFrame({
    "RecordID": ["R01", "R02", "R03", "R03", "R04"],
    "Gender": ["Male", "female", "M", "M", "F"],
    "Age": [24, 31, -5, -5, 28],
    "Height": [172, 1.65, 180, 180, 1.58],
    "HeightUnit": ["cm", "m", "cm", "cm", "m"]
})

display(dirty_demo)
clean_demo = dirty_demo.copy()

## TODO: Uncomment the lines below — standardize categories
# clean_demo["Gender"] = (
#     clean_demo["Gender"].str.strip().str.lower()
#     .replace({"m": "male", "f": "female"})
# )

## TODO: Uncomment the lines below — mark an impossible age as missing
# clean_demo.loc[clean_demo["Age"] < 0, "Age"] = np.nan

## TODO: Uncomment the lines below — convert all heights to centimetres
# clean_demo["Height_cm"] = np.where(
#     clean_demo["HeightUnit"].eq("m"),
#     clean_demo["Height"] * 100,
#     clean_demo["Height"]
# )

## TODO: Uncomment the lines below — remove exact duplicate records
# clean_demo = clean_demo.drop_duplicates()

## TODO: Uncomment the lines below — inspect the repaired table
# display(clean_demo)
# print("Remaining duplicate rows:", clean_demo.duplicated().sum())


> ### Interpretation — Warm-up dirty-data example
>
> **What the original table shows**
>
> - `Male`, `female`, `M`, and `F` are inconsistent labels for the same two categories.
> - `Age = -5` is impossible, so it should not be treated as a valid age.
> - Height is recorded in both metres and centimetres. Values cannot be compared until the units are standardized.
> - Record `R03` appears twice, so one copy is an exact duplicate.
>
> **What the cleaning code does**
>
> - Standardizes gender labels to `male` and `female`.
> - Converts impossible ages to `NaN` so they can be handled as missing values.
> - Converts every height to centimetres.
> - Removes the repeated row.
>
> **Main concept**
>
> Different data problems need different treatments. We do not use one command to “clean everything.”

## Task 1 – Audit the raw dataset
Produce a table containing data type, missing count, missing percentage and number of unique values for every column. Also report the number of exact duplicate rows.

The DataFrame structure is provided. Reveal one summary calculation at a time.

In [ ]:
summary = pd.DataFrame(index=df_raw.columns)
summary.index.name = "Column"

## TODO: Uncomment the lines below — data types
# summary["Data Type"] = df_raw.dtypes.astype(str)

## TODO: Uncomment the lines below — missing-value counts
# summary["Missing Count"] = df_raw.isna().sum()

## TODO: Uncomment the lines below — missing-value percentages
# summary["Missing Percentage"] = df_raw.isna().mean().mul(100)

## TODO: Uncomment the lines below — unique values
# summary["Unique Values"] = df_raw.nunique(dropna=False)

## TODO: Uncomment the lines below — sort the audit table
# summary = summary.sort_values("Missing Percentage", ascending=False)

display(summary)

## TODO: Uncomment the lines below — duplicate check
# print("Exact duplicate rows:", df_raw.duplicated().sum())


> ### Interpretation — Dataset audit
>
> **Expected output**
>
> - Dataset size: **891 rows × 12 columns**
> - `Cabin`: **687 missing values**, approximately **77.10%**
> - `Age`: **177 missing values**, approximately **19.87%**
> - `Embarked`: **2 missing values**, approximately **0.22%**
> - Exact duplicate rows: **0**
> - `PassengerId` and `Name` each have **891 unique values**
> - `Ticket` has **681 unique values**
>
> **What this means**
>
> - `Cabin` is too incomplete and too high-cardinality to use directly without careful transformation.
> - `Age` has enough missing values that deleting all affected rows would remove a substantial part of the dataset.
> - The two missing `Embarked` values can reasonably be handled using most-frequent imputation.
> - A duplicate check returning zero is still a valid and necessary result.
>
> **Main concept**
>
> We inspect the dataset before deciding how to clean it. The method depends on the type and amount of each problem.

## Task 2 – Interpret the problems
Answer in a Markdown cell:
1. Which columns contain missing values?
2. Which columns are identifiers or high-cardinality text?
3. Which columns require encoding?
4. Which numerical columns may require scaling?
5. Does the file contain exact duplicates?

### Suggested answer — Task 2

1. Missing values are present in `Age`, `Cabin`, and `Embarked`.
2. `PassengerId` is an identifier. `Name`, `Ticket`, and `Cabin` are high-cardinality text variables.
3. `Sex`, `Embarked`, and the engineered `Title` require categorical encoding. `Pclass` has an ordered interpretation, although treating it categorically is also reasonable.
4. `Age`, `Fare`/`LogFare`, `SibSp`, `Parch`, and `FamilySize` are numerical. `Age` and `Fare` especially differ greatly in scale.
5. The supplied file contains **zero exact duplicate rows**.

## Task 3 – Investigate Fare outliers
Calculate the IQR fences for `Fare`, count the flagged observations, display a box plot and explain why the flagged records should not automatically be removed.

The quartiles and box plot are active. Reveal the IQR rule and filtering logic.

In [ ]:
fare = df_raw["Fare"].dropna()

Q1 = fare.quantile(0.25)
Q3 = fare.quantile(0.75)

## TODO: Uncomment the lines below — calculate IQR and fences
# IQR = Q3 - Q1
# lower_fence = Q1 - 1.5 * IQR
# upper_fence = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)

## TODO: Uncomment the lines below — identify possible outliers
# fare_outlier_mask = (
#     (df_raw["Fare"] < lower_fence) |
#     (df_raw["Fare"] > upper_fence)
# )
#
# print("IQR:", IQR)
# print("Lower fence:", lower_fence)
# print("Upper fence:", upper_fence)
# print("Flagged observations:", int(fare_outlier_mask.sum()))

plt.figure(figsize=(8, 3.5))
plt.boxplot(fare, vert=False)
plt.xlabel("Fare")
plt.title("Fare box plot")
plt.tight_layout()
plt.show()

## TODO: Uncomment the lines below — inspect high-fare records
# display(
#     df_raw.loc[
#         df_raw["Fare"] > upper_fence,
#         ["Pclass", "Fare", "Cabin", "Survived"]
#     ].sort_values("Fare", ascending=False).head(10)
# )


> ### Interpretation — Fare outliers
>
> **Expected values**
>
> - \(Q_1 pprox 7.9104\)
> - \(Q_3 = 31.0000\)
> - \(IQR pprox 23.0896\)
> - Lower fence \(pprox -26.7240\)
> - Upper fence \(pprox 65.6344\)
> - Flagged observations: **116**
>
> **What the box plot shows**
>
> `Fare` is strongly right-skewed, with several values far above the upper IQR fence.
>
> **Why we do not delete all 116 records**
>
> The largest fares are mainly associated with first-class passengers and group tickets. They are unusual, but many are plausible rather than erroneous. The IQR rule identifies possible outliers; it does not prove that the values are wrong.
>
> **Decision used in this lab**
>
> Retain the valid observations and create `LogFare = log(1 + Fare)` to reduce skew and compress very large values.
>
> **Main concept**
>
> Outlier detection is an investigation step, not an automatic deletion rule.

## Task 4 – Compare Min–Max scaling and standardization

Use the same complete `Age` and `Fare` observations to compare the two representations.

- Min–Max scaling maps training values to a fixed range, commonly 0–1.
- Standardization centres values around 0 and scales them by the standard deviation.
- This is a demonstration only. The final pipeline will learn scaling parameters from `X_train`.


In [ ]:
scaling_demo = df_raw[["Age", "Fare"]].dropna().copy()
minmax_scaler = MinMaxScaler()
standard_scaler = StandardScaler()

## TODO: Uncomment the lines below — calculate Min–Max scaled values
# scaling_demo[["Age_MinMax", "Fare_MinMax"]] = (
#     minmax_scaler.fit_transform(scaling_demo[["Age", "Fare"]])
# )

## TODO: Uncomment the lines below — calculate standardized values
# scaling_demo[["Age_Standard", "Fare_Standard"]] = (
#     standard_scaler.fit_transform(scaling_demo[["Age", "Fare"]])
# )

## TODO: Uncomment the lines below — compare the representations
# display(scaling_demo.head(8))
# display(scaling_demo.describe().T)


> ### Interpretation — Min–Max versus standardization
>
> **Min–Max scaling**
>
> - Maps the minimum observed value to 0 and the maximum to 1.
> - Produces a bounded representation.
> - Is strongly affected by extreme values. The very large Titanic fares compress many normal fares close to zero.
>
> **Standardization**
>
> - Produces values measured in standard-deviation units.
> - Gives a mean close to 0 and a standard deviation close to 1.
> - Does not create a fixed range.
>
> **Why the final pipeline uses standardization**
>
> Logistic regression is scale-sensitive when regularization is used. Standardization places continuous numerical variables on comparable scales.
>
> **Important warning**
>
> This cell is only a visual demonstration. In the final model, scaling parameters are learned from the training set only.

## Task 5 – Compare ordinal and nominal encoding

`Pclass` has a meaningful order, whereas `Embarked` is nominal. Reveal the two encodings and discuss why arbitrary label numbers for `Embarked` would introduce a false order.


In [ ]:
encoding_demo = df_raw[["Pclass", "Embarked"]].head(8).copy()

## TODO: Uncomment the lines below — explicitly encode the ordered passenger class
# pclass_order = {1: 0, 2: 1, 3: 2}
# encoding_demo["Pclass_Ordinal"] = encoding_demo["Pclass"].map(pclass_order)

## TODO: Uncomment the lines below — one-hot encode the nominal embarkation port
# embarked_one_hot = pd.get_dummies(
#     encoding_demo["Embarked"],
#     prefix="Embarked",
#     dtype=int
# )
# encoding_demo = pd.concat([encoding_demo, embarked_one_hot], axis=1)

## TODO: Uncomment the lines below — inspect the encoded table
# display(encoding_demo)


> ### Interpretation — Ordinal and nominal encoding
>
> **`Pclass`**
>
> - Passenger class has an ordered interpretation: first, second, and third class.
> - An ordinal mapping can therefore be justified.
> - The mapping used here makes the ordering explicit; the exact numeric labels are not inherently meaningful.
>
> **`Embarked`**
>
> - `C`, `Q`, and `S` are port names with no natural ranking.
> - One-hot encoding creates separate binary columns and avoids inventing an order.
>
> **Why arbitrary label encoding would be misleading**
>
> Encoding `C = 0`, `Q = 1`, and `S = 2` could make a model treat `S` as greater than `Q` and assume numerical distances between ports.
>
> **Main concept**
>
> Choose an encoding method based on the meaning of the categories, not merely because the original values are text.

## Task 6 – Engineer features
Create:
- `FamilySize = SibSp + Parch + 1`
- `IsAlone`
- `CabinKnown`
- `Title` extracted from `Name`, grouping uncommon titles as `Rare`
- `LogFare = log(1 + Fare)`

The feature-engineering function and return statement are active. Reveal each new feature separately.

In [ ]:
def engineer_titanic_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()

    ## TODO: Uncomment the lines below — family-related features
    # result["FamilySize"] = result["SibSp"] + result["Parch"] + 1
    # result["IsAlone"] = (result["FamilySize"] == 1).astype(int)

    ## TODO: Uncomment the lines below — missing Cabin as information
    # result["CabinKnown"] = result["Cabin"].notna().astype(int)

    ## TODO: Uncomment the lines below — extract passenger title
    # result["Title"] = (
    #     result["Name"]
    #     .str.extract(r",\s*([^.]*)\.", expand=False)
    #     .str.strip()
    # )

    common_titles = {"Mr", "Miss", "Mrs", "Master"}

    ## TODO: Uncomment the lines below — group rare titles
    # result["Title"] = result["Title"].where(
    #     result["Title"].isin(common_titles),
    #     "Rare"
    # )

    ## TODO: Uncomment the lines below — reduce Fare skewness
    # result["LogFare"] = np.log1p(result["Fare"])

    return result


df = engineer_titanic_features(df_raw)
display(df.head())

## TODO: Uncomment the lines below — inspect engineered features
# display(
#     df[[
#         "Name", "Title", "SibSp", "Parch", "FamilySize",
#         "IsAlone", "Cabin", "CabinKnown", "Fare", "LogFare"
#     ]].head(10)
# )


> ### Interpretation — Engineered features
>
> **`FamilySize`**
>
> `SibSp + Parch + 1` includes the passenger and represents the total family group size.
>
> **`IsAlone`**
>
> Converts family size into a simple binary behavioural feature: travelling alone or not.
>
> **`CabinKnown`**
>
> Uses missingness itself as information. A recorded cabin may indirectly represent passenger class or access to better accommodation.
>
> **`Title`**
>
> Reduces the high-cardinality `Name` field into a small set of meaningful categories: `Mr`, `Mrs`, `Miss`, `Master`, and `Rare`.
>
> **`LogFare`**
>
> Reduces the influence of very large fares while preserving their ordering.
>
> **Main concept**
>
> Feature engineering transforms raw variables into representations that may be easier for a model to learn from. It is different from feature selection, which decides which variables to keep.

## Task 7 – Define X and y and split the data
Remove `PassengerId`, `Name`, `Ticket`, `Cabin` and the original `Fare`. Use a stratified 80:20 split with `random_state=42`.

The target and excluded-column list are provided. Reveal the separation and split.

In [ ]:
target = "Survived"
excluded_columns = ["PassengerId", "Name", "Ticket", "Cabin", "Fare"]

## TODO: Uncomment the lines below — separate predictors and target
# X = df.drop(columns=[target] + excluded_columns)
# y = df[target]

## TODO: Uncomment the lines below — stratified train-test split
# X_train, X_test, y_train, y_test = train_test_split(
#     X,
#     y,
#     test_size=0.20,
#     random_state=42,
#     stratify=y
# )

## TODO: Uncomment the lines below — verify the split
# print("Training shape:", X_train.shape)
# print("Test shape:", X_test.shape)
# display(
#     y_train.value_counts(normalize=True)
#     .rename("Training Proportion")
#     .to_frame()
# )


> ### Interpretation — Train–test split
>
> **Expected shapes**
>
> - Full predictor matrix: **891 rows × 11 features**
> - Training set: **712 rows**
> - Test set: **179 rows**
>
> **Expected training target proportions**
>
> - Did not survive: approximately **61.66%**
> - Survived: approximately **38.34%**
>
> **Why `stratify=y` is used**
>
> It preserves approximately the same survival proportions in the training and test sets.
>
> **Why some columns are removed**
>
> - `PassengerId`: identifier
> - `Name`, `Ticket`, `Cabin`: high-cardinality raw text
> - `Fare`: replaced by the less-skewed `LogFare`
>
> **Main concept**
>
> The split must happen before fitting imputers, scalers, encoders, or feature-selection methods. Otherwise, information from the test set leaks into training.

## Task 8 – Examine correlation and feature redundancy

Correlation is most appropriate for numerical–numerical relationships. Here it is used mainly to inspect redundancy among predictors using the training partition only. High correlation between two inputs does not prove that either one should automatically be removed.


In [ ]:
correlation_features = [
    "Age", "Fare" if "Fare" in X_train.columns else "LogFare",
    "SibSp", "Parch", "FamilySize", "LogFare"
]
correlation_features = list(dict.fromkeys(
    feature for feature in correlation_features if feature in X_train.columns
))

## TODO: Uncomment the lines below — calculate the training-set correlation matrix
# correlation_matrix = X_train[correlation_features].corr()
# display(correlation_matrix.round(3))

## TODO: Uncomment the lines below — list strong predictor-to-predictor correlations
# upper_triangle = correlation_matrix.where(
#     np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
# )
# strong_pairs = (
#     upper_triangle.stack()
#     .rename_axis(index=["Feature 1", "Feature 2"])
#     .reset_index(name="Correlation")
# )
# strong_pairs = strong_pairs[
#     strong_pairs["Correlation"].abs() >= 0.70
# ].sort_values("Correlation", key=lambda s: s.abs(), ascending=False)
# display(strong_pairs)


> ### Interpretation — Correlation and redundancy
>
> **Important expected correlations**
>
> - `SibSp` with `FamilySize`: approximately **0.874**
> - `Parch` with `FamilySize`: approximately **0.789**
>
> **Why these relationships are strong**
>
> `FamilySize` is calculated directly from `SibSp` and `Parch`, so it naturally repeats some of their information.
>
> **What this demonstrates**
>
> A feature can be meaningful but still be redundant. Highly correlated predictors may increase complexity without adding much new information.
>
> **Do not conclude automatically**
>
> High correlation does not prove that one feature must be deleted. The final decision should also consider model performance, interpretation, and domain knowledge.

## Task 9 – Apply ANOVA on training data
Use median imputation fitted only to the numerical training features. Rank `Age`, `SibSp`, `Parch`, `FamilySize` and `LogFare` using ANOVA F-scores.

The feature list and imputer are active. Reveal fitting, scoring and ranking.

In [ ]:
numerical_features = ["Age", "SibSp", "Parch", "FamilySize", "LogFare"]
num_imputer = SimpleImputer(strategy="median")

## TODO: Uncomment the lines below — fit imputation on training data only
# X_train_num = num_imputer.fit_transform(X_train[numerical_features])

## TODO: Uncomment the lines below — calculate ANOVA scores
# f_scores, p_values = f_classif(X_train_num, y_train)

## TODO: Uncomment the lines below — build and rank the results table
# anova_results = pd.DataFrame({
#     "Feature": numerical_features,
#     "F Score": f_scores,
#     "p-value": p_values
# }).sort_values("F Score", ascending=False)
#
# display(anova_results)


> ### Interpretation — ANOVA F-test
>
> **Expected ranking**
>
> 1. `LogFare`: \(F pprox 102.44,\ p pprox 1.39 	imes 10^{-22}\)
> 2. `Parch`: \(F pprox 5.07,\ p pprox 0.0247\)
> 3. `Age`: \(F pprox 3.78,\ p pprox 0.0523\)
> 4. `FamilySize`: \(F pprox 0.52,\ p pprox 0.4725\)
> 5. `SibSp`: \(F pprox 0.48,\ p pprox 0.4866\)
>
> **How to interpret this**
>
> - `LogFare` shows the strongest difference in mean values between the survival classes.
> - `Parch` shows weaker but statistically significant mean separation at the 0.05 level.
> - `Age` is close to, but slightly above, the conventional 0.05 threshold.
> - `FamilySize` and `SibSp` show weak univariate mean separation.
>
> **Important warning**
>
> A low p-value does not guarantee that the feature will improve the final model. ANOVA evaluates each feature individually and mainly focuses on differences in class means.

## Task 10 – Rank numerical features using Mutual Information

Mutual Information estimates how much knowing a feature reduces uncertainty about the target. Unlike Pearson correlation, it can capture more general dependencies. Compare its ranking with ANOVA rather than assuming the two methods must agree.


In [ ]:
# Reuse X_train_num created by the ANOVA step.

## TODO: Uncomment the lines below — calculate Mutual Information scores
# mi_scores = mutual_info_classif(
#     X_train_num,
#     y_train,
#     random_state=42
# )

## TODO: Uncomment the lines below — build and compare the ranking table
# mi_results = pd.DataFrame({
#     "Feature": numerical_features,
#     "Mutual Information": mi_scores
# }).sort_values("Mutual Information", ascending=False)
#
# display(mi_results)
# display(
#     anova_results[["Feature", "F Score"]]
#     .merge(mi_results, on="Feature", how="outer")
#     .sort_values("F Score", ascending=False)
# )


> ### Interpretation — Mutual Information
>
> **Expected ranking**
>
> 1. `LogFare`: approximately **0.153**
> 2. `FamilySize`: approximately **0.073**
> 3. `Age`: approximately **0.038**
> 4. `Parch`: approximately **0.012**
> 5. `SibSp`: approximately **0.009**
>
> **Why this differs from ANOVA**
>
> ANOVA evaluates separation of class means. Mutual Information can detect more general and potentially nonlinear dependency.
>
> `FamilySize` is the clearest example: it has a weak ANOVA score but a noticeably higher Mutual Information score. Its relationship with survival may not be represented well by a simple difference in average family size.
>
> **Main concept**
>
> Different feature-selection methods ask different questions, so their rankings do not have to agree.

## Task 11 – Apply Chi-square on training data
Test `Pclass`, `Sex`, `Embarked`, `Title`, `CabinKnown` and `IsAlone` against `Survived`. Display the statistic and p-value for each feature.

The loop and feature list are prepared. Reveal the contingency table, test and result collection.

In [ ]:
categorical_features = [
    "Pclass", "Sex", "Embarked", "Title", "CabinKnown", "IsAlone"
]

chi_rows = []

for feature in categorical_features:
    feature_values = X_train[feature].astype("object").fillna("Missing")

    ## TODO: Uncomment the lines below — create the contingency table
    # contingency_table = pd.crosstab(feature_values, y_train)

    ## TODO: Uncomment the lines below — apply Chi-square
    # chi2_stat, p_value, dof, expected = chi2_contingency(
    #     contingency_table
    # )

    ## TODO: Uncomment the lines below — store the result
    # chi_rows.append({
    #     "Feature": feature,
    #     "Chi-square": chi2_stat,
    #     "p-value": p_value,
    #     "Degrees of Freedom": dof,
    #     "Minimum Expected Count": expected.min()
    # })

## TODO: Uncomment the lines below — rank and display all categorical features
# chi_results = pd.DataFrame(chi_rows).sort_values("p-value")
# display(chi_results)
#
# print("Example contingency table: Sex vs Survived")
# display(pd.crosstab(X_train["Sex"], y_train, margins=True))


> ### Interpretation — Chi-square tests
>
> **Expected strength of association**
>
> The strongest categorical relationships with `Survived` are generally:
>
> 1. `Title`
> 2. `Sex`
> 3. `Pclass` and `CabinKnown`
> 4. `IsAlone`
> 5. `Embarked`
>
> All six features normally produce p-values below 0.05 in this training split.
>
> **Example: Sex versus survival**
>
> - Female: **65 did not survive, 188 survived**
> - Male: **374 did not survive, 85 survived**
>
> The observed counts differ strongly from what would be expected if sex and survival were independent.
>
> **Assumption warning**
>
> `Embarked` includes a very small missing-value group, giving a minimum expected count below 5. Note that the usual Chi-square approximation is less reliable when expected cell counts are very small.
>
> **Main concept**
>
> A small p-value indicates evidence of association, not causation and not guaranteed predictive improvement.

## Task 12 – Build the complete pipeline
Use median imputation and standardization for numerical columns, and most-frequent imputation plus one-hot encoding for categorical columns. Train logistic regression.

The pipeline structure is active. Reveal the preprocessing steps and final fitting line.

In [ ]:
def build_pipeline(num_features, cat_features):
    numeric_pipeline = Pipeline([
        ## TODO: Uncomment the lines below — numerical preprocessing
        # ("imputer", SimpleImputer(strategy="median")),
        # ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ## TODO: Uncomment the lines below — categorical preprocessing
        # ("imputer", SimpleImputer(strategy="most_frequent")),
        # ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, num_features),
        ("categorical", categorical_pipeline, cat_features)
    ])

    model_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(max_iter=2000, random_state=42)
        )
    ])

    return model_pipeline


all_feature_model = build_pipeline(
    numerical_features,
    categorical_features
)

## TODO: Uncomment the lines below — fit the complete pipeline
# all_feature_model.fit(X_train, y_train)
# print("The all-feature pipeline was fitted successfully.")


> ### Interpretation — Complete preprocessing pipeline
>
> **Numerical branch**
>
> 1. Replace missing values using the training-set median.
> 2. Standardize using the training-set mean and standard deviation.
>
> **Categorical branch**
>
> 1. Replace missing values using the training-set most frequent category.
> 2. One-hot encode categories.
> 3. Ignore unseen future categories instead of raising an error.
>
> **Why the pipeline matters**
>
> The same learned transformations are applied consistently to training, validation, test, and future data. It also prevents accidental preprocessing leakage during cross-validation.
>
> **Expected message**
>
> `The all-feature pipeline was fitted successfully.`

## Task 13 – Evaluate and compare

### 9.1 Evaluate the all-feature model
The prediction and reporting structure are active. Reveal the metric calculations and model call.

In [ ]:
def evaluate_classifier(model, X_eval, y_eval, label):
    predictions = model.predict(X_eval)
    results = {"Model": label}

    ## TODO: Uncomment the lines below — calculate evaluation metrics
    # results["Accuracy"] = accuracy_score(y_eval, predictions)
    # results["Precision"] = precision_score(
    #     y_eval, predictions, zero_division=0
    # )
    # results["Recall"] = recall_score(
    #     y_eval, predictions, zero_division=0
    # )
    # results["F1 Score"] = f1_score(
    #     y_eval, predictions, zero_division=0
    # )

    print(label)
    print(classification_report(y_eval, predictions, digits=4))

    ConfusionMatrixDisplay.from_predictions(y_eval, predictions)
    plt.title(f"Confusion matrix – {label}")
    plt.tight_layout()
    plt.show()

    return results


## TODO: Uncomment the lines below — evaluate the all-feature model
# all_results = evaluate_classifier(
#     all_feature_model,
#     X_test,
#     y_test,
#     "All valid features"
# )


> ### Interpretation — All-feature model
>
> **Expected test results**
>
> - Accuracy: approximately **0.832**
> - Precision: **0.800**
> - Recall: approximately **0.754**
> - F1-score: approximately **0.776**
>
> **Expected confusion matrix**
>
> - True negatives: **97**
> - False positives: **13**
> - False negatives: **17**
> - True positives: **52**
>
> **How to explain the metrics**
>
> - Accuracy: about 83.2% of all passengers are classified correctly.
> - Precision: 80% of predicted survivors actually survived.
> - Recall: the model identifies about 75.4% of the actual survivors.
> - F1-score balances precision and recall.
>
> **Main concept**
>
> Accuracy alone does not show which type of error the model makes. The confusion matrix and class-specific metrics provide the missing detail.

### 13.2 Select a reduced feature set
Use the top three ANOVA-ranked numerical features and the categorical features with Chi-square `p < 0.05`.

In [ ]:
## TODO: Uncomment the lines below — select numerical features
# selected_numeric = anova_results.head(3)["Feature"].tolist()

## TODO: Uncomment the lines below — select categorical features
# selected_categorical = chi_results.loc[
#     chi_results["p-value"] < 0.05,
#     "Feature"
# ].tolist()
#
# if not selected_categorical:
#     selected_categorical = chi_results.head(2)["Feature"].tolist()

## TODO: Uncomment the lines below — combine and inspect the selected features
# selected_features = selected_numeric + selected_categorical
# print("Selected numerical features:", selected_numeric)
# print("Selected categorical features:", selected_categorical)


> ### Interpretation — Selected features
>
> **Expected numerical selection**
>
> The top three ANOVA-ranked numerical features are:
>
> - `LogFare`
> - `Parch`
> - `Age`
>
> **Expected categorical selection**
>
> All six tested categorical features normally have \(p < 0.05\):
>
> - `Title`
> - `Sex`
> - `CabinKnown`
> - `Pclass`
> - `IsAlone`
> - `Embarked`
>
> **What was removed**
>
> `SibSp` and `FamilySize` are excluded by the ANOVA top-three rule.
>
> **Important observation**
>
> This is a statistically reduced feature set, not necessarily the best predictive feature set.

### 13.3 Train and compare the reduced model
Build the same leakage-safe pipeline using only the selected features.

In [ ]:
## TODO: Uncomment the lines below — build and fit the reduced model
# selected_model = build_pipeline(
#     selected_numeric,
#     selected_categorical
# )
#
# selected_model.fit(X_train[selected_features], y_train)

## TODO: Uncomment the lines below — evaluate the reduced model
# selected_results = evaluate_classifier(
#     selected_model,
#     X_test[selected_features],
#     y_test,
#     "Statistically reduced features"
# )

## TODO: Uncomment the lines below — compare test-set metrics
# comparison = pd.DataFrame([
#     all_results,
#     selected_results
# ]).set_index("Model")
#
# display(comparison)


> ### Interpretation — Reduced-model comparison
>
> **Expected reduced-model test results**
>
> - Accuracy: approximately **0.816**
> - Precision: approximately **0.773**
> - Recall: approximately **0.739**
> - F1-score: approximately **0.756**
>
> **Expected confusion matrix**
>
> - True negatives: **95**
> - False positives: **15**
> - False negatives: **18**
> - True positives: **51**
>
> **Comparison with the all-feature model**
>
> The reduced model is slightly worse on all four metrics. The all-feature model has an F1-score of about 0.776, while the reduced model has about 0.756.
>
> **Correct conclusion**
>
> Statistical screening simplified the model, but it did not improve predictive performance in this experiment.
>
> **Main concept**
>
> Statistical significance and predictive usefulness are not the same. Features that appear weak individually may still contribute when combined with other variables.

### 13.4 Optional: compare cross-validation F1 scores
Use this part only if time permits.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## TODO: Uncomment the lines below — cross-validate both pipelines
# all_cv = cross_val_score(
#     all_feature_model,
#     X,
#     y,
#     cv=cv,
#     scoring="f1"
# )
#
# selected_cv = cross_val_score(
#     selected_model,
#     X[selected_features],
#     y,
#     cv=cv,
#     scoring="f1"
# )

## TODO: Uncomment the lines below — summarize cross-validation
# cv_summary = pd.DataFrame({
#     "Model": ["All valid features", "Statistically reduced features"],
#     "Mean CV F1": [all_cv.mean(), selected_cv.mean()],
#     "CV F1 SD": [all_cv.std(), selected_cv.std()]
# }).set_index("Model")
#
# display(cv_summary)


> ### Interpretation — Cross-validation
>
> **Expected five-fold F1 summary**
>
> - All valid features: mean F1 approximately **0.769**, SD approximately **0.024**
> - Reduced features: mean F1 approximately **0.754**, SD approximately **0.016**
>
> **Interpretation**
>
> The all-feature model remains slightly better across multiple training-validation splits. The reduced model varies a little less, but the difference in average performance does not support claiming that feature selection improved prediction.
>
> **Main concept**
>
> Cross-validation gives a more stable comparison than relying on a single train–test split, but the untouched test set is still needed for final evaluation.

## Reflection
1. Why was the median preferred for numerical imputation?
2. Why is direct one-hot encoding of `Name` unsuitable?
3. Why should an IQR flag trigger investigation rather than automatic deletion?
4. How do Min–Max scaling and standardization differ?
5. Why is one-hot encoding appropriate for `Embarked`, while an ordered mapping can be justified for `Pclass`?
6. Which feature pairs appear redundant in the correlation analysis?
7. Why can ANOVA and Mutual Information produce different feature rankings?
8. Did feature selection improve prediction, simplify the model, or both?
9. Which operations would leak information if fitted before the train–test split?


## Suggested answers to the reflection questions

1. **Why median imputation?**  
   The median is less affected by skewness and extreme values than the mean.

2. **Why not one-hot encode the full `Name` column?**  
   `Name` has very high cardinality and would create many sparse, passenger-specific columns. Extracting `Title` gives a smaller and more meaningful feature.

3. **Why not delete every IQR outlier?**  
   High fares may be valid first-class or group-ticket fares. IQR identifies unusual values, not confirmed errors.

4. **Min–Max versus standardization?**  
   Min–Max maps values to a fixed interval such as 0–1. Standardization expresses values in standard-deviation units around a mean close to zero.

5. **Why different encodings for `Embarked` and `Pclass`?**  
   `Embarked` has no natural order, so one-hot encoding avoids false ranking. `Pclass` has a meaningful order, so ordinal encoding can be justified.

6. **Which features are redundant?**  
   `FamilySize` is constructed from `SibSp` and `Parch`, so strong correlations are expected. `LogFare` is also a transformation of `Fare`.

7. **Why do ANOVA and Mutual Information disagree?**  
   ANOVA focuses on class-mean separation. Mutual Information detects broader dependency, including nonlinear patterns.

8. **Did feature selection improve the model?**  
   It simplified the feature set but slightly reduced predictive performance.

9. **What would cause leakage?**  
   Calculating imputation values, scaling parameters, category encodings, or feature rankings using the full dataset before the train–test split.

# Final message to give the class

> A statistically significant feature is not automatically a strong predictive feature.  
> The final feature set should combine **statistical evidence, model performance, domain knowledge, data availability, interpretability, and practical constraints**.